# Path D: Dynamic/Stochastic Phenotyping — DS5 Step 2

**Notebook:** `106_ds5_dynamic_stochastic_phenotype.ipynb`  
**Stage:** Stage 4, Step 2 — Path D (sixth and final Step 2 path)  
**Date:** 2026-04-20  
**Dataset:** DS5 (N=414 StoP-eligible participants)

---

## Central Question

> After Paths A, C, F, E, and B — does Path D reveal a **genuinely new phenotype structure** along the dynamic/stochastic axis, or does its main value lie in **deepening and interpreting the behavioral-dynamic taxonomy** already identified?

**Pre-analysis answer:** Path D is NOT expected to reveal a genuinely new phenotype structure. The behavioral-dynamic axis (sig_hat × H_emp) was already Path A's primary structure. Remaining dynamic features either fail the correlation gate, have too many missing values, or have near-zero discriminatory power. Path D's thesis value: model-language grounding, chi_behav characterization, DyS regime validation.

See blueprint: `outputs/reports/stage4/ds5_path_d_dynamic_stochastic_blueprint.md`


## Section 0: Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.metrics import calinski_harabasz_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# --- Constants ---
RANDOM_STATE = 42
N_BOOTSTRAPS = 100
SAMPLE_FRAC = 0.80
MIN_CELL = 30
JACCARD_STABLE = 0.75
JACCARD_MARG = 0.60

# Intended feature sets (blueprint-specified)
INTENDED_D1 = ['sig_hat', 'H_emp', 'chi_behav']  # Primary: StoP diffusion + entropy + StM susceptibility
INTENDED_D2 = ['sig_hat', 'H_emp', 'mu_hat']     # Secondary: Pure StoP parameters
INTENDED_D3 = ['sig_hat', 'H_emp', 'H_hour']     # Sensitivity: + temporal distribution

# Paths
DATA_DIR = Path('outputs/model_outputs/phenotyping')
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Color palettes
CLUSTER_COLORS = {
    'D_sde1': '#2196F3',  'D_sde2': '#FF9800',
    'D_sde3': '#F44336',  'D_sde4': '#9C27B0',
}
CONDITION_COLORS = {'control': '#4CAF50', 'default': '#FF9800', 'reintervention': '#2196F3'}
PATH_A_COLORS = {'C1': '#2196F3', 'C2': '#FF9800', 'C3': '#F44336'}

np.random.seed(RANDOM_STATE)
print("Setup complete.")


## Load Data

In [ ]:
# Load feature matrix
fm = pd.read_csv(DATA_DIR / 'ds5_phenotype_feature_matrix.csv')

# Load cross-path labels
path_a = pd.read_csv(DATA_DIR / 'ds5_behav_only_cluster_assignments.csv')[
    ['participant_id_canonical', 'cluster_behav', 'profile_p1']]
path_f = pd.read_csv(DATA_DIR / 'ds5_hierarchical_assignments.csv')[
    ['participant_id_canonical', 'cell_f']]
path_e = pd.read_csv(DATA_DIR / 'ds5_condition_cluster_assignments.csv')[
    ['participant_id_canonical', 'cond_cluster_full']]
path_b = pd.read_csv(DATA_DIR / 'ds5_intervention_cluster_assignments.csv')[
    ['participant_id_canonical', 'cluster_b_full']]

# Merge
df = fm.merge(path_a, on='participant_id_canonical', how='left')
df = df.merge(path_f, on='participant_id_canonical', how='left')
df = df.merge(path_e, on='participant_id_canonical', how='left')
df = df.merge(path_b, on='participant_id_canonical', how='left')

# Handle duplicate chi_behav column (pandas renames to chi_behav.1)
if 'chi_behav.1' in df.columns:
    df = df.drop(columns=['chi_behav.1'])
df = df.loc[:, ~df.columns.duplicated()]

print(f"N={len(df)} participants loaded.")
print(f"Condition distribution:\n{df['study_condition'].value_counts()}")
print(f"\nPath A clusters:\n{df['cluster_behav'].value_counts()}")


## Helper Functions

In [ ]:
def fast_jaccard(X, k, n_bootstraps=N_BOOTSTRAPS, sample_frac=SAMPLE_FRAC,
                  random_state=RANDOM_STATE):
    """Vectorized bootstrap Jaccard stability."""
    rng = np.random.RandomState(random_state)
    n = len(X)
    km_full = KMeans(n_clusters=k, random_state=random_state, n_init=20).fit(X)
    labels_full = km_full.labels_
    jaccard_scores = []
    for i in range(n_bootstraps):
        idx = rng.choice(n, size=int(n * sample_frac), replace=False)
        km_sub = KMeans(n_clusters=k, random_state=random_state + i, n_init=20).fit(X[idx])
        labels_sub = km_sub.labels_
        mapping = {}
        used_sub = set()
        for c in range(k):
            mask = labels_full[idx] == c
            if mask.sum() == 0: continue
            counts = np.bincount(labels_sub[mask], minlength=k)
            for best in np.argsort(-counts):
                if best not in used_sub:
                    mapping[c] = best; used_sub.add(best); break
        ls_aligned = np.array([mapping.get(l, l) for l in labels_sub])
        lf = labels_full[idx]
        same_full = lf[:, None] == lf[None, :]
        same_sub = ls_aligned[:, None] == ls_aligned[None, :]
        upper = np.triu_indices(len(idx), k=1)
        a = np.sum(same_full[upper] & same_sub[upper])
        b = np.sum(same_full[upper] ^ same_sub[upper])
        jaccard_scores.append(a / (a + b) if (a + b) > 0 else 0.0)
    return float(np.mean(jaccard_scores))

def fit_ordered_km(X_scaled, k, prefix, order_col_idx=0):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(X_scaled)
    order = np.argsort(km.cluster_centers_[:, order_col_idx])
    mapping = {old: new for new, old in enumerate(order)}
    labels = np.array([mapping[l] for l in km.labels_])
    return labels, [f'{prefix}{i+1}' for i in range(k)]

def flag_stability(j):
    if j is None: return 'REJECTED_MINCELL'
    return 'STABLE' if j >= JACCARD_STABLE else 'MARGINAL' if j >= JACCARD_MARG else 'UNSTABLE'

def kselection_loop(X_scaled, analysis_label, N, k_range=None, prefix='D_sde'):
    if k_range is None: k_range = [2, 3, 4]
    results = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(X_scaled)
        min_cell = int(np.bincount(km.labels_).min())
        if min_cell < MIN_CELL:
            results.append({'analysis': analysis_label, 'N': N, 'k_tested': k,
                             'sil': None, 'min_cell': min_cell, 'jaccard': None,
                             'flag': 'REJECTED_MINCELL', 'selected': False})
            continue
        sil = silhouette_score(X_scaled, km.labels_)
        jaccard = fast_jaccard(X_scaled, k)
        results.append({'analysis': analysis_label, 'N': N, 'k_tested': k,
                         'sil': round(sil, 4), 'min_cell': min_cell,
                         'jaccard': round(jaccard, 4), 'flag': flag_stability(jaccard),
                         'selected': False})
    valid = [r for r in results if r['flag'] in ('STABLE', 'MARGINAL') and r['sil']]
    if valid: max(valid, key=lambda r: r['sil'])['selected'] = True
    return results

print("Helper functions defined.")


## Section 1: Dynamic Feature Audit

In [ ]:
AUDIT_FEATURES = ['sig_hat', 'H_emp', 'x_mean', 'mu_hat', 'H_SDE', 'beta_behav',
                   'F_behav', 'chi_behav', 'H_hour', 'H_app']
AUDIT_FEATURES = [f for f in AUDIT_FEATURES if f in df.columns]

print("Missingness:")
for f in AUDIT_FEATURES:
    print(f"  {f}: N={df[f].notna().sum()}, NaN={df[f].isna().sum()}")

corr_matrix = df[AUDIT_FEATURES].corr()
corr_matrix.round(3)


In [ ]:
# --- fig_d1: Feature correlation matrix ---
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 8})
n = len(AUDIT_FEATURES)
for i in range(n):
    for j in range(n):
        if i != j and abs(corr_matrix.iloc[i, j]) > 0.70:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=2.5))
ax.set_title('Dynamic Feature Correlation Matrix\n(black border = |r| > 0.70 — gate fires)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_d1_feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_d1_feature_correlations.png")


In [ ]:
# Apply correlation gate
print("--- CORRELATION GATE RESULTS ---")
print(f"H_SDE vs sig_hat:   r={corr_matrix.loc['H_SDE','sig_hat']:.3f} → GATE FIRES → H_SDE DROPPED")
print(f"F_behav vs x_mean:  r={corr_matrix.loc['F_behav','x_mean']:.3f} → GATE FIRES → F_behav DROPPED")
print(f"beta_behav (derived from sig_hat, β=2/σ²) → EXCLUDED")
print(f"chi_behav vs sig_hat: r={corr_matrix.loc['chi_behav','sig_hat']:.3f} → passes")
print(f"chi_behav vs H_emp:   r={corr_matrix.loc['chi_behav','H_emp']:.3f} → passes")
print(f"mu_hat vs sig_hat:    r={corr_matrix.loc['mu_hat','sig_hat']:.3f} → passes")

FEATURES_D1 = ['sig_hat', 'H_emp', 'chi_behav']
FEATURES_D2 = ['sig_hat', 'H_emp', 'mu_hat']
h_hour_std = df['H_hour'].std()
FEATURES_D3 = ['sig_hat', 'H_emp', 'H_hour'] if h_hour_std >= 0.20 else None
RUN_D3 = h_hour_std >= 0.20

print(f"\nH_hour std={h_hour_std:.4f} → {'D3 will run' if RUN_D3 else 'D3 SKIPPED (std < 0.20)'}")
print(f"\nD1 (primary): {FEATURES_D1}")
print(f"D2 (secondary): {FEATURES_D2}")
print(f"D3 (sensitivity): {FEATURES_D3}")


## Section 2: mu_hat and sigma_mu_ratio Descriptive

In [ ]:
# mu_hat analysis
print("--- mu_hat by Path A cluster ---")
for c in ['C1', 'C2', 'C3']:
    sub = df[df['cluster_behav'] == c]['mu_hat']
    nz = (sub.abs() < 0.005).sum()
    print(f"  {c} (N={len(sub)}): mean={sub.mean():.5f}, std={sub.std():.5f}, |μ|<0.005: {nz} ({100*nz/len(sub):.1f}%)")

mu_kw = stats.kruskal(*[df[df['cluster_behav']==c]['mu_hat'].values for c in ['C1','C2','C3']])
print(f"  KW test: H={mu_kw.statistic:.3f}, p={mu_kw.pvalue:.4f}")
print("  NOTE: C1 has 45.2% near-zero mu_hat — DS5 is diffusion-dominated, not drift-dominated.")

# sigma_mu_ratio
ratio_df = df[df['sigma_mu_ratio'].notna()].copy()
print(f"\n--- sigma_mu_ratio (N={len(ratio_df)}, {100*len(ratio_df)/len(df):.1f}% available) ---")
print(f"  mean={ratio_df['sigma_mu_ratio'].mean():.2f}, median={ratio_df['sigma_mu_ratio'].median():.2f}")
print(f"  Noise-dominated (>10): {(ratio_df['sigma_mu_ratio']>10).mean()*100:.1f}%")
for c in ['C1','C2','C3']:
    sub = ratio_df[ratio_df['cluster_behav']==c]
    print(f"  {c}: N={len(sub)}, mean_ratio={sub['sigma_mu_ratio'].mean():.2f}")
print("  CAUTION: C1 underrepresented (45% have |μ|<0.005 — excluded from subsample)")

# chi_behav (KEY FINDING)
print("\n--- chi_behav by Path A cluster (CRITICAL FINDING) ---")
for c in ['C1','C2','C3']:
    sub = df[df['cluster_behav']==c]['chi_behav']
    print(f"  {c}: mean={sub.mean():.2f}, std={sub.std():.2f}, median={sub.median():.2f}")
chi_kw = stats.kruskal(*[df[df['cluster_behav']==c]['chi_behav'].values for c in ['C1','C2','C3']])
print(f"  KW: H={chi_kw.statistic:.3f}, p={chi_kw.pvalue:.2e}")
print("  KEY: C1 (low-noise, low-override) has HIGHEST chi_behav = 56.83")
print("  This is counterintuitive: low-noise participants are MOST susceptible to tipping.")
print("  Because chi_behav = (2/σ²)·x_mean·(1-x_mean) — small σ → large β → high χ")


In [ ]:
# fig_d2: Feature distributions by Path A cluster
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
features_to_plot = ['sig_hat', 'H_emp', 'chi_behav', 'mu_hat', 'F_behav', 'H_SDE']
labels_order = ['C1', 'C2', 'C3']
for ax, feat in zip(axes.flat, features_to_plot):
    data_list = [df[df['cluster_behav']==c][feat].dropna().values for c in labels_order]
    vp = ax.violinplot(data_list, positions=range(3), showmedians=True)
    for body, color in zip(vp['bodies'], [PATH_A_COLORS[c] for c in labels_order]):
        body.set_facecolor(color); body.set_alpha(0.7)
    ax.set_xticks(range(3)); ax.set_xticklabels(labels_order)
    ax.set_title(feat, fontweight='bold')
axes[0, 2].set_title('chi_behav\n(★ C1 = highest susceptibility)', fontweight='bold', color='#C62828')
fig.suptitle('Dynamic/Stochastic Feature Distributions by Path A Cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_d2_feature_distributions_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

# fig_d3: Drift/noise profiles
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df['mu_hat'], bins=40, color='#546E7A', alpha=0.7, edgecolor='white')
axes[0].axvline(-0.005, color='red', ls='--', alpha=0.7, label='|μ|=0.005')
axes[0].axvline(0.005, color='red', ls='--', alpha=0.7)
axes[0].set_title(f'mu_hat (N={len(df)})\n|μ|<0.005: {(df["mu_hat"].abs()<0.005).sum()} participants', fontweight='bold')
axes[0].legend()
axes[1].hist(ratio_df['sigma_mu_ratio'], bins=40, color='#37474F', alpha=0.7, edgecolor='white')
axes[1].axvline(10, color='orange', ls='--', label='ratio=10')
axes[1].set_title(f'σ̂/|μ| (N={len(ratio_df)} subsample)\nNoise-dominated (>10): {(ratio_df["sigma_mu_ratio"]>10).sum()}', fontweight='bold')
axes[1].legend()
fig.suptitle('Drift vs Noise Dominance', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_d3_drift_noise_profiles.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 3: Primary Clustering — D1 [sig_hat, H_emp, chi_behav]

In [ ]:
X_d1 = df[FEATURES_D1].values
scaler_d1 = StandardScaler()
X_d1_scaled = scaler_d1.fit_transform(X_d1)

stability_records = []
print("Running k-selection for D1...")
d1_results = kselection_loop(X_d1_scaled, 'D1_full_sample', N=len(df))
stability_records.extend(d1_results)

for r in d1_results:
    print(f"  k={r['k_tested']}: sil={r['sil']}, min_cell={r['min_cell']}, "
          f"J={r['jaccard']}, {r['flag']}, selected={r['selected']}")

d1_sel = next((r for r in d1_results if r['selected']), None)
K_D1 = d1_sel['k_tested'] if d1_sel else None
print(f"\nD1 selected k={K_D1}" if K_D1 else "\nD1: No stable solution — regime analysis only")


In [ ]:
# fig_d4: K-selection curves
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
k_vals = [r['k_tested'] for r in d1_results]
sil_v = [r['sil'] if r['sil'] else np.nan for r in d1_results]
jac_v = [r['jaccard'] if r['jaccard'] else np.nan for r in d1_results]
mc_v  = [r['min_cell'] for r in d1_results]

axes[0].plot(k_vals, sil_v, 'o-', color='#1565C0', lw=2, ms=8)
axes[0].set_title('Silhouette', fontweight='bold'); axes[0].set_xticks(k_vals)

axes[1].plot(k_vals, jac_v, 's-', color='#2E7D32', lw=2, ms=8)
axes[1].axhline(JACCARD_STABLE, color='green', ls='--', alpha=0.7, label=f'Stable ({JACCARD_STABLE})')
axes[1].axhline(JACCARD_MARG, color='orange', ls='--', alpha=0.7, label=f'Marginal ({JACCARD_MARG})')
axes[1].set_title('Jaccard Stability', fontweight='bold'); axes[1].legend(fontsize=8); axes[1].set_xticks(k_vals)

axes[2].bar(k_vals, mc_v, color=['#E53935' if v < MIN_CELL else '#43A047' for v in mc_v])
axes[2].axhline(MIN_CELL, color='red', ls='--', alpha=0.7, label=f'MIN_CELL={MIN_CELL}')
axes[2].set_title('Min Cell Size', fontweight='bold'); axes[2].legend(fontsize=8); axes[2].set_xticks(k_vals)

if K_D1:
    for ax in axes: ax.axvline(K_D1, color='purple', ls=':', alpha=0.8, lw=2)

fig.suptitle('D1 [sig_hat, H_emp, chi_behav] — K-Selection', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_d4_kselection_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Fit D1 final solution
if K_D1:
    labels_d1, label_names_d1 = fit_ordered_km(X_d1_scaled, K_D1, prefix='D_sde')
    df['cluster_d_full'] = [label_names_d1[l] for l in labels_d1]
    print("D1 cluster distribution:")
    print(df['cluster_d_full'].value_counts())

    # Profile table
    profile_cols = ['sig_hat','H_emp','chi_behav','mu_hat','x_mean','F_behav','H_SDE','H_hour']
    profile_cols = [c for c in profile_cols if c in df.columns]
    profile_data = {}
    for name in label_names_d1:
        sub = df[df['cluster_d_full']==name]
        profile_data[name] = {'N': len(sub)}
        for col in profile_cols:
            profile_data[name][f'{col}_mean'] = sub[col].mean()
            profile_data[name][f'{col}_std'] = sub[col].std()
    profiles_df = pd.DataFrame(profile_data).T

    print("\nD1 cluster profiles (primary features + overlays):")
    display_cols = ['N'] + [f'{c}_mean' for c in ['sig_hat','H_emp','chi_behav','mu_hat','x_mean']]
    print(profiles_df[display_cols].round(4))
else:
    df['cluster_d_full'] = 'undifferentiated'
    label_names_d1 = []
    profiles_df = pd.DataFrame()
    print("No stable D1 solution — regime analysis only.")


## Section 4: Cross-Path ARI Comparisons

In [ ]:
ari_records = []
if K_D1 and len(label_names_d1) > 1:
    d1_labels = df['cluster_d_full'].values

    comparisons = [
        ('D1 vs Path A (cluster_behav)', df['cluster_behav'].values, 'HIGH → D1 recovers Path A' if True else ''),
        ('D1 vs Phase 1 (profile_p1)', df['profile_p1'].values, 'Expected LOW — NeT-independent'),
        ('D1 vs Path F (cell_f)', df['cell_f'].values, 'Hierarchical alignment'),
        ('D1 vs Path E (cond_cluster_full)', df['cond_cluster_full'].values, 'Condition-structure alignment'),
        ('D1 vs Path B (cluster_b_full)', df['cluster_b_full'].values, 'Friction-response alignment'),
    ]
    for label, ref_labels, desc in comparisons:
        ari = adjusted_rand_score(ref_labels, d1_labels)
        interp = ('HIGH — recovers Path A' if ari > 0.70 else
                  'MODERATE — partial reorganization' if ari > 0.40 else 'LOW — new structure')
        ari_records.append({'comparison': label, 'ARI': round(ari, 3), 'interpretation': interp})
        print(f"  {label}: ARI={ari:.3f} — {interp}")

    print("\nD1 vs Path A cross-tabulation:")
    print(pd.crosstab(df['cluster_d_full'], df['cluster_behav']))


## Section 5: Secondary Clustering — D2 [sig_hat, H_emp, mu_hat]

In [ ]:
X_d2 = df[FEATURES_D2].values
scaler_d2 = StandardScaler()
X_d2_scaled = scaler_d2.fit_transform(X_d2)

d2_results = kselection_loop(X_d2_scaled, 'D2_full_sample', N=len(df))
stability_records.extend(d2_results)

for r in d2_results:
    print(f"  k={r['k_tested']}: sil={r['sil']}, min_cell={r['min_cell']}, J={r['jaccard']}, {r['flag']}")

d2_sel = next((r for r in d2_results if r['selected']), None)
if d2_sel:
    K_D2 = d2_sel['k_tested']
    labels_d2, names_d2 = fit_ordered_km(X_d2_scaled, K_D2, prefix='D2_')
    df['cluster_d2'] = [names_d2[l] for l in labels_d2]
    ari_d2 = adjusted_rand_score(df['cluster_behav'].values, df['cluster_d2'].values)
    ari_records.append({'comparison': 'D2 vs Path A', 'ARI': round(ari_d2, 3),
                         'interpretation': f'mu_hat contribution (expected > 0.90 if no structure added)'})
    print(f"\nD2 k={K_D2}, ARI(D2 vs Path A)={ari_d2:.3f}")
    print("CONFIRMED: mu_hat adds no new structure." if ari_d2 > 0.90 else
          f"NOTE: ARI={ari_d2:.3f} — mu_hat partially reorganizes space.")
else:
    df['cluster_d2'] = 'N/A'
    print("D2: No stable solution.")


## Section 6: Sensitivity — D3 [sig_hat, H_emp, H_hour]

In [ ]:
if RUN_D3:
    X_d3 = df[FEATURES_D3].values
    X_d3_scaled = StandardScaler().fit_transform(X_d3)
    d3_results = kselection_loop(X_d3_scaled, 'D3_sensitivity', N=len(df))
    for r in d3_results: r['analysis'] = 'D3_sensitivity'
    stability_records.extend(d3_results)
    for r in d3_results:
        print(f"  k={r['k_tested']}: sil={r['sil']}, min_cell={r['min_cell']}, J={r['jaccard']}, {r['flag']}")
    d3_sel = next((r for r in d3_results if r['selected']), None)
    if d3_sel:
        labels_d3, names_d3 = fit_ordered_km(X_d3_scaled, d3_sel['k_tested'], prefix='D3_')
        df['cluster_d3'] = [names_d3[l] for l in labels_d3]
        ari_d3 = adjusted_rand_score(df['cluster_behav'].values, df['cluster_d3'].values)
        ari_records.append({'comparison': 'D3 vs Path A (H_hour sensitivity)', 'ARI': round(ari_d3, 3),
                             'interpretation': 'H_hour contribution'})
        print(f"  D3 k={d3_sel['k_tested']}, ARI={ari_d3:.3f}")
    else:
        df['cluster_d3'] = 'N/A'
else:
    print(f"D3 SKIPPED — H_hour std={h_hour_std:.4f} (< 0.20 threshold not met)")
    stability_records.append({'analysis':'D3_sensitivity','N':len(df),'k_tested':None,
                               'sil':None,'min_cell':None,'jaccard':None,
                               'flag':'SKIPPED_LOW_VARIANCE','selected':False})
    df['cluster_d3'] = 'N/A'


## Section 7: Detailed Profile Analysis

In [ ]:
if K_D1 and len(label_names_d1) > 1:
    # fig_d5: Cluster profiles
    profile_features = ['sig_hat','H_emp','chi_behav','mu_hat','x_mean']
    fig, axes = plt.subplots(1, len(profile_features), figsize=(4*len(profile_features), 5))
    for ax, feat in zip(axes, profile_features):
        means = [df[df['cluster_d_full']==c][feat].mean() for c in label_names_d1]
        stds  = [df[df['cluster_d_full']==c][feat].std() for c in label_names_d1]
        colors_list = [CLUSTER_COLORS.get(c, '#607D8B') for c in label_names_d1]
        ax.bar(range(len(label_names_d1)), means, yerr=stds, color=colors_list, alpha=0.8,
               capsize=5, edgecolor='white')
        ax.set_xticks(range(len(label_names_d1)))
        ax.set_xticklabels(label_names_d1, rotation=30, ha='right')
        ax.set_title(feat, fontweight='bold')
    fig.suptitle('D1 Cluster Profiles', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_d5_cluster_profiles.png', dpi=150, bbox_inches='tight')
    plt.show()

    # fig_d6: chi_behav distributions
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (label_set, label_col, title) in zip(axes, [
        (['C1','C2','C3'], 'cluster_behav', 'chi_behav by Path A Cluster\n★ C1 (low-noise) = highest susceptibility'),
        (label_names_d1, 'cluster_d_full', 'chi_behav by D1 Cluster\n(ordered by sig_hat ascending)')]):
        data_list = [df[df[label_col]==c]['chi_behav'].values for c in label_set]
        vp = ax.violinplot(data_list, positions=range(len(label_set)), showmedians=True)
        color_map = PATH_A_COLORS if label_col == 'cluster_behav' else CLUSTER_COLORS
        for body, c in zip(vp['bodies'], label_set):
            body.set_facecolor(color_map.get(c, '#607D8B')); body.set_alpha(0.7)
        ax.set_xticks(range(len(label_set)))
        ax.set_xticklabels([f'{c}\nN={len(df[df[label_col]==c])}' for c in label_set])
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel('chi_behav')
        for i, d in enumerate(data_list):
            ax.annotate(f'μ={np.mean(d):.1f}', xy=(i, np.mean(d)), ha='center', va='bottom',
                       fontsize=10, fontweight='bold')
    fig.suptitle('Tipping-Point Proximity (chi_behav)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_d6_chi_behav_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    # fig_d7: cross-path heatmap
    crosstab = pd.crosstab(df['cluster_d_full'], df['cluster_behav'], normalize='index')
    fig, ax = plt.subplots(figsize=(7, 4))
    d1_a_ari = next((r['ARI'] for r in ari_records if 'D1 vs Path A' in r['comparison']), None)
    sns.heatmap(crosstab, annot=True, fmt='.2f', cmap='Blues', ax=ax, linewidths=0.5, vmin=0, vmax=1)
    ax.set_title(f'D1 vs Path A (row-normalized)\nARI = {d1_a_ari}', fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig_d7_crosspath_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()


## Section 8: DyS Regime Flag Analysis

In [ ]:
# is_dys_eligible × Path A
ct1 = pd.crosstab(df['cluster_behav'], df['is_dys_eligible'])
chi2_1, p1, dof1, _ = stats.chi2_contingency(ct1)
print(f"is_dys_eligible × Path A: χ²={chi2_1:.3f}, p={p1:.2e}")
print("Proportion DyS-eligible:")
for c in ['C1','C2','C3']:
    sub = df[df['cluster_behav']==c]
    print(f"  {c}: {sub['is_dys_eligible'].mean():.3f} ({sub['is_dys_eligible'].sum()}/{len(sub)})")

# is_dys_eligible × condition
ct2 = pd.crosstab(df['study_condition'], df['is_dys_eligible'])
chi2_2, p2, _, _ = stats.chi2_contingency(ct2)
print(f"\nis_dys_eligible × condition: χ²={chi2_2:.3f}, p={p2:.2e}")
print("Proportion DyS-eligible by condition:")
for cond in ['control','default','reintervention']:
    sub = df[df['study_condition']==cond]
    print(f"  {cond}: {sub['is_dys_eligible'].mean():.3f} ({sub['is_dys_eligible'].sum()}/{len(sub)})")
print("\nKEY: Default has fewest deterministic participants — extends Path E homogenization finding.")

# behaviorally_deterministic
ct3 = pd.crosstab(df['study_condition'], df['behaviorally_deterministic'])
chi2_3, p3, _, _ = stats.chi2_contingency(ct3)
print(f"\nbehaviorally_deterministic × condition: χ²={chi2_3:.3f}, p={p3:.2e}")


In [ ]:
# fig_d8: DyS regime by cluster and condition
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
w = 0.35

ax = axes[0]
patha_clusters = ['C1','C2','C3']
x = np.arange(len(patha_clusters))
dys_p = [df[df['cluster_behav']==c]['is_dys_eligible'].mean() for c in patha_clusters]
det_p = [df[df['cluster_behav']==c]['behaviorally_deterministic'].mean() for c in patha_clusters]
ax.bar(x-w/2, dys_p, w, label='DyS-eligible', color='#1E88E5', alpha=0.8)
ax.bar(x+w/2, det_p, w, label='Behaviorally deterministic', color='#004D40', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(patha_clusters)
ax.set_title('DyS Flags by Path A Cluster', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0, 1)

ax = axes[1]
conds = ['control','default','reintervention']
x = np.arange(3)
dys_c = [df[df['study_condition']==c]['is_dys_eligible'].mean() for c in conds]
det_c = [df[df['study_condition']==c]['behaviorally_deterministic'].mean() for c in conds]
ax.bar(x-w/2, dys_c, w, label='DyS-eligible', color='#1E88E5', alpha=0.8)
ax.bar(x+w/2, det_c, w, label='Behaviorally deterministic', color='#004D40', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(conds)
ax.set_title(f'DyS Flags by Condition\n(χ²={chi2_2:.1f}, p={p2:.2e})', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0, 1)

fig.suptitle('DyS Stochastic Regime Classification', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_d8_dys_regime_by_cluster_condition.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 9: StM Model-Language Landscape

In [ ]:
# fig_d9: Stochastic landscape
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for c, color in PATH_A_COLORS.items():
    sub = df[df['cluster_behav']==c]
    ax.scatter(sub['sig_hat'], sub['H_emp'], c=color, alpha=0.6,
               s=20 + sub['chi_behav']/3, label=f'{c} (N={len(sub)})', edgecolors='none')
ax.set_xlabel('sig_hat (σ̂ — diffusion)'); ax.set_ylabel('H_emp (entropy)')
ax.set_title('StoP Space by Path A Cluster\n(size ∝ chi_behav)', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
if K_D1 and len(label_names_d1) > 1:
    for c in label_names_d1:
        sub = df[df['cluster_d_full']==c]
        ax.scatter(sub['sig_hat'], sub['H_emp'], c=CLUSTER_COLORS.get(c,'#607D8B'), alpha=0.6,
                   s=20+sub['chi_behav']/3, label=f'{c} (N={len(sub)})', edgecolors='none')
    ax.set_title(f'StoP Space by D1 Cluster (k={K_D1})\n(size ∝ chi_behav)', fontweight='bold')
else:
    ax.set_title('D1 clustering not feasible — Path A overlay', fontweight='bold')
    for c, color in PATH_A_COLORS.items():
        sub = df[df['cluster_behav']==c]
        ax.scatter(sub['sig_hat'], sub['H_emp'], c=color, alpha=0.4, s=20, edgecolors='none')
ax.set_xlabel('sig_hat'); ax.set_ylabel('H_emp')
ax.legend(fontsize=8)

fig.suptitle('Behavioral Stochastic Landscape (sig_hat × H_emp × chi_behav)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_d9_stochastic_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

# Tipping-point rank
top30 = df.nlargest(30, 'chi_behav')
print("Top 30 chi_behav participants by Path A cluster:")
print(top30['cluster_behav'].value_counts())


## Section 10: Condition Sensitivity

In [ ]:
# Within-reintervention D1
df_rei = df[df['study_condition']=='reintervention'].copy()
X_rei = df_rei[FEATURES_D1].values
X_rei_scaled = StandardScaler().fit_transform(X_rei)
rei_results = kselection_loop(X_rei_scaled, 'D1_reintervention', N=len(df_rei))
for r in rei_results: r['analysis'] = 'D1_reintervention'
stability_records.extend(rei_results)
for r in rei_results:
    print(f"  rei k={r['k_tested']}: min_cell={r['min_cell']}, J={r['jaccard']}, {r['flag']}")

rei_sel = next((r for r in rei_results if r['selected']), None)
if rei_sel:
    labels_rei, names_rei = fit_ordered_km(X_rei_scaled, rei_sel['k_tested'], prefix='REI_D')
    df_rei['cluster_d_rei'] = [names_rei[l] for l in labels_rei]
    ari_rei = adjusted_rand_score(df_rei['cluster_behav'].values, labels_rei)
    ari_records.append({'comparison': 'D1 within-rei vs Path A', 'ARI': round(ari_rei, 3),
                         'interpretation': 'Reintervention-specific alignment'})
    df = df.merge(df_rei[['participant_id_canonical','cluster_d_rei']],
                   on='participant_id_canonical', how='left')
    df['cluster_d_rei'] = df['cluster_d_rei'].fillna('N/A')
    print(f"  rei k={rei_sel['k_tested']}, ARI vs Path A = {ari_rei:.3f}")
else:
    df['cluster_d_rei'] = 'N/A'
    print("  Within-rei: No stable solution.")

# KW H_out_resid by D1 cluster
if K_D1 and len(label_names_d1) > 1:
    kw_h = stats.kruskal(*[df[df['cluster_d_full']==c]['H_out_resid'].values for c in label_names_d1])
    print(f"\nKW H_out_resid by D1 cluster: H={kw_h.statistic:.3f}, p={kw_h.pvalue:.4f}")
    print("(Cross-check Path B: behavioral type predicts friction-response diversity)")


## Section 11: Psychosocial Overlays

In [ ]:
psych_file = Path('outputs/features/ds5/ds5_survey_features.csv')
if psych_file.exists() and K_D1 and len(label_names_d1) > 1:
    psych = pd.read_csv(psych_file)
    id_cols = ['participant_id_canonical', 'study_condition']
    scale_cols = [c for c in psych.columns if c not in id_cols
                  and pd.api.types.is_numeric_dtype(psych[c])][:8]
    df_psych = df.merge(psych[['participant_id_canonical'] + scale_cols],
                         on='participant_id_canonical', how='left')
    alpha_b = 0.05 / max(len(scale_cols), 1)
    print(f"Bonferroni α = {alpha_b:.4f} (n={len(scale_cols)} tests)")
    for scale in scale_cols:
        groups = [df_psych[df_psych['cluster_d_full']==c][scale].dropna().values for c in label_names_d1]
        if all(len(g) > 0 for g in groups):
            try:
                kw = stats.kruskal(*groups)
                sig = "SIGNIFICANT" if kw.pvalue < alpha_b else "ns"
                print(f"  {scale}: H={kw.statistic:.3f}, p={kw.pvalue:.4f} [{sig}]")
            except Exception as e:
                print(f"  {scale}: {e}")
else:
    print("Psychosocial overlays: participant-level file not available or no D1 clustering.")


## Section 12: Save Outputs

In [ ]:
# Save assignments CSV
assignment_cols = ['participant_id_canonical','study_condition','profile_p1',
                    'cluster_d_full','cluster_d_rei','cluster_behav','cell_f',
                    'cond_cluster_full','cluster_b_full']
assignment_cols = [c for c in assignment_cols if c in df.columns]
df[assignment_cols].to_csv(DATA_DIR / 'ds5_dynamic_cluster_assignments.csv', index=False)
print(f"Saved assignments (N={len(df)})")

# Save profiles CSV
if len(profiles_df) > 0:
    profiles_df.to_csv(DATA_DIR / 'ds5_dynamic_cluster_profiles.csv')
    print(f"Saved profiles ({len(profiles_df)} clusters)")
else:
    pd.DataFrame({'note': ['Regime analysis only — no stable D1 clustering']}).to_csv(
        DATA_DIR / 'ds5_dynamic_cluster_profiles.csv', index=False)

# Save stability CSV
stability_df = pd.DataFrame(stability_records)
stability_df.to_csv(DATA_DIR / 'ds5_dynamic_cluster_stability.csv', index=False)
print(f"Saved stability ({len(stability_df)} rows)")
print("\nAll outputs saved to outputs/model_outputs/phenotyping/")


## Verification

In [ ]:
import os
csv_out = list(DATA_DIR.glob('ds5_dynamic*.csv')) + list(DATA_DIR.glob('ds5_dynamic*.md'))
csv_out += [DATA_DIR / 'model_run_summary_path_d.md']
fig_out = list(FIG_DIR.glob('fig_d*.png'))

print(f"Output files ({len(csv_out)}):")
for f in sorted(csv_out): print(f"  {f.name}")

print(f"\nFigures ({len(fig_out)}):")
for f in sorted(fig_out): print(f"  {f.name}")

asgn = pd.read_csv(DATA_DIR / 'ds5_dynamic_cluster_assignments.csv')
print(f"\nAssignments shape: {asgn.shape}")
print(f"cluster_d_full:\n{asgn['cluster_d_full'].value_counts()}")
print(f"\nNaN in cluster_d_full: {asgn['cluster_d_full'].isna().sum()}")
